In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
np.set_printoptions(precision=2, suppress=True)

In [3]:
ratings = pd.read_csv(
    r"C:\Users\user\Downloads\ml-100k\ml-100k\u.data",
    sep="\t",
    names=["user_id" , "movie_id", "rating" , "timestamp"]
)

train,test = train_test_split(ratings, test_size=0.2, random_state=42)

In [4]:
rating_matrix = ratings.pivot(index="user_id", columns="movie_id", values="rating")
all_users = ratings["user_id"].unique()
all_movies = ratings["movie_id"].unique()

In [5]:
train_matrix = train.pivot(index="user_id",columns="movie_id",values="rating").reindex(index=all_users,columns=all_movies)
train_matrix.shape

(943, 1682)

In [6]:
test_matrix = test.pivot(index="user_id",columns="movie_id",values="rating").reindex(index=all_users,columns=all_movies)
test_matrix.shape

(943, 1682)

In [7]:
def split_Y_R(train_matrix,test_matrix):
    
    Y_train = np.array(train_matrix.copy())
    R_train = Y_train.copy()
    
    nan_mask = np.isnan(Y_train)
    
    Y_train[nan_mask] = 0
    R_train[nan_mask] = 0
    R_train[~nan_mask] = 1
    
    Y_test = np.array(test_matrix.copy())
    R_test = Y_test.copy()
    
    nan_mask = np.isnan(Y_test)
    
    Y_test[nan_mask] = 0
    R_test[nan_mask] = 0
    R_test[~nan_mask] = 1

    return Y_train, Y_test, R_train, R_test

In [8]:
film_num = all_movies.shape[0]
user_num = all_users.shape[0]

In [9]:
def costFunc(W, X, B_film, B_user,yArray, rArray):
    error = predict(W, X, B_film, B_user) - yArray
    error = rArray * error
    cost = np.sum(np.square(error)) / 2
    return cost

In [10]:
def predict(W, X, B_film, B_user, global_mean):
    prediction_matrix = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:] + global_mean
    return prediction_matrix

In [11]:
def calculate_Global_Mean(Y,R):
    global_mean = np.sum(Y) / np.sum(R)
    return global_mean

In [12]:
np.array([1,2,5])[np.newaxis,:]

array([[1, 2, 5]])

In [13]:
def gradientDescent(W, X, B_film, B_user, global_mean, yArray, rArray, iteration, learning_rate, momentum = 0.8 ,lambda_ = 0.1):

    
    dW = None; dX = None; dB_film = None; dB_user = None;
    
    vW = np.zeros_like(W)
    vX = np.zeros_like(X)
    vB_film = np.zeros_like(B_film)
    vB_user = np.zeros_like(B_user)
        
    for i in range(iteration + 1):
        
        prediction = predict(W, X, B_film, B_user, global_mean)
        error = (prediction - yArray) * rArray
        
        dW = np.dot(error, X) + lambda_ * W
        dX = np.dot(error.T, W) + lambda_ * X
        
        dB_film = np.sum(error, axis = 0)
        dB_user = np.sum(error, axis = 1)

        vW = momentum * vW + learning_rate * dW
        vX = momentum * vX + learning_rate * dX
        vB_film = momentum * vB_film + learning_rate * dB_film
        vB_user = momentum * vB_user + learning_rate * dB_user
        
        W = W - vW
        X = X - vX
        B_film = B_film - vB_film
        B_user = B_user - vB_user

        if i % 100 == 0:
            data_loss = np.sum( np.square(error) / 2)
            regularization_loss = (lambda_ / 2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
            total_loss = data_loss + regularization_loss
            #print(i,"th iteration     Loss:", "{:.4f}".format(total_loss))

    return W, X, B_film, B_user

In [14]:
def calculate_errors(W, X, B_film, B_user, global_mean, yArray, rArray):
    n = np.sum(rArray)
    prediction = predict(W, X, B_film, B_user, global_mean)
    error = (prediction - yArray) * rArray        
    
    mse = np.sum(np.square(error)) / n
    rmse = np.sqrt(mse)
    mae = np.sum(np.abs(error)) / n

    return "{:.4f}".format(mse)

In [15]:
Y_train, Y_test, R_train, R_test = split_Y_R(train_matrix, test_matrix)
global_mean = calculate_Global_Mean(Y_train, R_train)

In [24]:
feature_num = 3
W = np.random.randn(user_num,feature_num) * 0.01
X = np.random.randn(film_num,feature_num) * 0.01
B_film = np.zeros((film_num,))
B_user = np.zeros((user_num,))
W , X, B_film, B_user = gradientDescent(W, X, B_film, B_user, global_mean, Y_train, R_train, 
                                        iteration=500, learning_rate=0.005, momentum=0.85, lambda_= 1)

In [25]:
print("Train")
print(calculate_errors(W, X, B_film, B_user, global_mean, Y_train, R_train))

Train
0.6340


In [26]:
print("Test")
print(calculate_errors(W, X, B_film, B_user, global_mean, Y_test, R_test))

Test
0.8822


In [27]:
def selectHyperParameter(user_num, film_num, global_mean, Y_train, Y_test, R_train, R_test):
    np.random.seed(42)
    learingRateList = [0.0005, 0.001, 0.002]
    featureList= [3, 5 , 10]
    lambdaList = [1, 0.5, 0.1 , 0.01, 0.05]
    print("Feature | Rate   | Lambda | Train RMSE | Test RMSE")
    print("------------------------------------------------")
    
    for i in learingRateList:
        for j in featureList:
            for l in lambdaList:
                feature_num = j
                W = np.random.randn(user_num,feature_num) * 0.01
                X = np.random.randn(film_num,feature_num) * 0.01
                B_film = np.zeros((film_num,))
                B_user = np.zeros((user_num,))
                W , X, B_film, B_user = gradientDescent(W, X, B_film, B_user, global_mean, Y_train, R_train, 
                                                        iteration=500, learning_rate= i, momentum=0.85, lambda_= l)
                errorTrain = calculate_errors(W, X, B_film, B_user, global_mean, Y_train, R_train)
                errorTest = calculate_errors(W, X, B_film, B_user, global_mean, Y_test, R_test)
                
                print(j.ljust(7) ,"| ", i.ljust(7), "| ",l.ljust(7), "| ", errorTrain.ljust(7), "| ",errorTest.ljust(7))
                

In [23]:
selectHyperParameter(user_num, film_num , global_mean, Y_train, Y_test, R_train, R_test)

Feature | Rate   | Lambda | Train RMSE | Test RMSE
------------------------------------------------
3        |  0.0005     |  1      |  0.6382      |  0.8854
3        |  0.0005     |  0.5      |  0.6358      |  0.8855
3        |  0.0005     |  0.1      |  0.6381      |  0.9245
3        |  0.0005     |  0.01      |  0.6396      |  0.9374
3        |  0.0005     |  0.05      |  0.6358      |  0.9136
5        |  0.0005     |  1      |  0.5686      |  0.9265
5        |  0.0005     |  0.5      |  0.5615      |  0.9597
5        |  0.0005     |  0.1      |  0.5583      |  0.9807
5        |  0.0005     |  0.01      |  0.5563      |  1.0006
5        |  0.0005     |  0.05      |  0.5567      |  1.0174
10        |  0.0005     |  1      |  0.4247      |  1.0484
10        |  0.0005     |  0.5      |  0.4189      |  1.1280
10        |  0.0005     |  0.1      |  0.4149      |  1.1898
10        |  0.0005     |  0.01      |  0.4114      |  1.2161
10        |  0.0005     |  0.05      |  0.4128      |  1.